# MAESTRO MIDI song-string databases and matching

This notebook uses the expanded MAESTRO dataset prepared in `maestro_train` and `maestro_songs`. The training directory contains the longest recording for each normalized canonical song, and the grouped song directory contains every MIDI recording under its canonical-song directory.

The notebook extracts full-song highest-note base strings, builds one lookup database per string-processing approach, matches every grouped query recording against those databases, and reports compression, accuracy, and runtime results.


In [19]:
from collections import Counter
from pathlib import Path
import base64
import zlib
import mido
import pandas as pd

TRAIN_DIR = Path("maestro_train")
SONGS_DIR = Path("maestro_songs")
ONSET_GROUP_SECONDS = 0.08
PIANO_LOWEST_MIDI_NOTE = 21
PIANO_HIGHEST_MIDI_NOTE = 108
PRINTABLE_ASCII = [chr(code) for code in range(33, 127)]


In [20]:
def extract_note_onsets(path):
    """Extract note-on events from a MIDI file with absolute times in seconds.

    Args:
        path: Path to a MIDI file.

    Returns:
        A list of dictionaries, each containing the onset time in seconds, MIDI note number, note name, and velocity.
    """
    midi = mido.MidiFile(path)
    tempo = 500_000
    seconds = 0.0
    note_onsets = []

    for message in mido.merge_tracks(midi.tracks):
        seconds += mido.tick2second(message.time, midi.ticks_per_beat, tempo)

        if message.type == "set_tempo":
            tempo = message.tempo
        elif message.type == "note_on" and message.velocity > 0:
            note_onsets.append(
                {
                    "time": seconds,
                    "note": message.note,
                    "velocity": message.velocity,
                }
            )

    return note_onsets


In [21]:
def highest_notes_by_onset_group(note_onsets, onset_group_seconds=ONSET_GROUP_SECONDS):
    """Select the highest note from each onset group across an entire song.

    Args:
        note_onsets: Note-on dictionaries returned by `extract_note_onsets`.
        onset_group_seconds: Maximum time span from the first onset in a group for notes to be treated as the same time step.

    Returns:
        A list of MIDI note numbers, one highest note per onset group for the full song.
    """
    if not note_onsets:
        return []

    start_time = note_onsets[0]["time"]
    groups = []
    current_group = []
    group_start = None

    for note in note_onsets:
        aligned_time = note["time"] - start_time
        if not current_group or aligned_time - group_start <= onset_group_seconds:
            if not current_group:
                group_start = aligned_time
            current_group.append(note)
        else:
            groups.append(current_group)
            current_group = [note]
            group_start = aligned_time

    if current_group:
        groups.append(current_group)

    return [max(group, key=lambda onset: onset["note"])["note"] for group in groups]


In [22]:
def build_piano_note_character_map(lowest_note=PIANO_LOWEST_MIDI_NOTE, highest_note=PIANO_HIGHEST_MIDI_NOTE):
    """Build a deterministic one-character ASCII encoding for piano MIDI notes.

    Args:
        lowest_note: Lowest supported MIDI note number. Defaults to A0, the lowest piano key.
        highest_note: Highest supported MIDI note number. Defaults to C8, the highest piano key.

    Returns:
        A dictionary mapping each supported MIDI note number to a unique printable ASCII character.
    """
    supported_notes = list(range(lowest_note, highest_note + 1))
    if len(supported_notes) > len(PRINTABLE_ASCII):
        raise ValueError("The requested note range needs more printable ASCII characters than are available")
    return {note: PRINTABLE_ASCII[index] for index, note in enumerate(supported_notes)}


In [23]:
def encode_notes_as_ascii(notes, note_character_map):
    """Encode a note list as a base string string.

    Args:
        notes: Sequence of MIDI note numbers.
        note_character_map: Mapping from MIDI note number to unique ASCII character.

    Returns:
        A string containing one ASCII character per note and no separators.
    """
    unsupported_notes = sorted({note for note in notes if note not in note_character_map})
    if unsupported_notes:
        raise ValueError(f"Unsupported notes outside the ASCII map: {unsupported_notes}")
    return "".join(note_character_map[note] for note in notes)


In [24]:
def midi_file_to_note_string(path, note_character_map):
    """Convert one MIDI file into the full-song highest-note ASCII string.

    Args:
        path: Path to a MIDI file.
        note_character_map: Mapping from MIDI note number to unique ASCII character.

    Returns:
        A base string for the full song.
    """
    note_onsets = extract_note_onsets(path)
    highest_notes = highest_notes_by_onset_group(note_onsets)
    return encode_notes_as_ascii(highest_notes, note_character_map)


In [25]:
def lowest_notes_by_onset_group(note_onsets, onset_group_seconds=ONSET_GROUP_SECONDS):
    """Select the lowest note from each onset group across an entire song.

    Args:
        note_onsets: Note-on dictionaries returned by `extract_note_onsets`.
        onset_group_seconds: Maximum time span from the first onset in a group for
            notes to be treated as the same time step.

    Returns:
        A list of MIDI note numbers, one lowest note per onset group for the full song.
    """
    if not note_onsets:
        return []

    start_time = note_onsets[0]["time"]
    groups = []
    current_group = []
    group_start = None

    for note in note_onsets:
        aligned_time = note["time"] - start_time
        if not current_group or aligned_time - group_start <= onset_group_seconds:
            if not current_group:
                group_start = aligned_time
            current_group.append(note)
        else:
            groups.append(current_group)
            current_group = [note]
            group_start = aligned_time

    if current_group:
        groups.append(current_group)

    return [min(group, key=lambda onset: onset["note"])["note"] for group in groups]


In [26]:
def top_k_notes_by_onset_group(note_onsets, k, onset_group_seconds=ONSET_GROUP_SECONDS):
    """Select the top-k highest notes from each onset group across an entire song.

    For each onset group up to k notes are selected, encoded ascending by pitch so
    the resulting flat list interleaves lower and upper voices in a predictable order.
    This typically produces a string up to k times longer than the single-note string.

    Args:
        note_onsets: Note-on dictionaries returned by `extract_note_onsets`.
        k: Maximum number of notes to select per onset group.
        onset_group_seconds: Maximum time span from the first onset in a group for
            notes to be treated as the same time step.

    Returns:
        A flat list of MIDI note numbers, up to k per onset group, sorted ascending
        by pitch within each group.
    """
    if not note_onsets:
        return []

    start_time = note_onsets[0]["time"]
    groups = []
    current_group = []
    group_start = None

    for note in note_onsets:
        aligned_time = note["time"] - start_time
        if not current_group or aligned_time - group_start <= onset_group_seconds:
            if not current_group:
                group_start = aligned_time
            current_group.append(note)
        else:
            groups.append(current_group)
            current_group = [note]
            group_start = aligned_time

    if current_group:
        groups.append(current_group)

    result = []
    for group in groups:
        top_notes = sorted(group, key=lambda onset: onset["note"], reverse=True)[:k]
        result.extend(note["note"] for note in sorted(top_notes, key=lambda o: o["note"]))
    return result


In [27]:
def midi_file_to_note_string_extraction(path, note_character_map, extraction="highest"):
    """Convert a MIDI file to a note string using a named extraction method.

    Args:
        path: Path to a MIDI file.
        note_character_map: Mapping from MIDI note number to unique ASCII character.
        extraction: Extraction name. One of ``'highest'``, ``'lowest'``,
            ``'top2'``, or ``'top3'``.

    Returns:
        A separator-free ASCII note string for the requested extraction.
    """
    note_onsets = extract_note_onsets(path)
    if extraction == "highest":
        notes = highest_notes_by_onset_group(note_onsets)
    elif extraction == "lowest":
        notes = lowest_notes_by_onset_group(note_onsets)
    elif extraction == "top2":
        notes = top_k_notes_by_onset_group(note_onsets, k=2)
    elif extraction == "top3":
        notes = top_k_notes_by_onset_group(note_onsets, k=3)
    else:
        raise ValueError(f"Unknown extraction type: {extraction!r}. Choose from 'highest', 'lowest', 'top2', 'top3'.")
    return encode_notes_as_ascii(notes, note_character_map)


In [28]:
def repeated_run_signature(note_string, min_run_length=2):
    """Keep only repeated-character runs and collapse each kept run to one character.

    Args:
        note_string: Separator-free ASCII note string.
        min_run_length: Minimum adjacent run length required for a character to be kept.

    Returns:
        A string containing one character for each adjacent run whose length is at least `min_run_length`.
    """
    if min_run_length <= 0:
        raise ValueError("min_run_length must be positive")
    if not note_string:
        return ""

    signature = []
    current_character = note_string[0]
    count = 1

    for character in note_string[1:]:
        if character == current_character:
            count += 1
        else:
            if count >= min_run_length:
                signature.append(current_character)
            current_character = character
            count = 1

    if count >= min_run_length:
        signature.append(current_character)
    return "".join(signature)


In [29]:
def run_length_encode(note_string):
    """Encode repeated adjacent characters as character-count runs.

    Args:
        note_string: Separator-free ASCII note string.

    Returns:
        A run-length encoded string where a single character is unchanged and a run is stored as character plus decimal count.
    """
    if not note_string:
        return ""

    encoded = []
    current_character = note_string[0]
    count = 1

    for character in note_string[1:]:
        if character == current_character:
            count += 1
        else:
            encoded.append(current_character if count == 1 else f"{current_character}{count}")
            current_character = character
            count = 1

    encoded.append(current_character if count == 1 else f"{current_character}{count}")
    return "".join(encoded)


In [30]:
def collapse_repeated_chunks(note_string, chunk_size):
    """Collapse adjacent repeated chunks of a fixed length to one copy.

    Args:
        note_string: Separator-free ASCII note string.
        chunk_size: Number of characters in each chunk to compare.

    Returns:
        A string where immediately repeated chunks of `chunk_size` are represented once.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    output = []
    previous_chunk = None
    index = 0

    while index < len(note_string):
        chunk = note_string[index : index + chunk_size]
        if chunk != previous_chunk:
            output.append(chunk)
            previous_chunk = chunk
        index += chunk_size

    return "".join(output)


In [31]:
def collapse_repeated_motifs(note_string, min_chunk_size=2, max_chunk_size=8):
    """Collapse adjacent repeated motifs using several chunk sizes.

    Args:
        note_string: Separator-free ASCII note string.
        min_chunk_size: Smallest motif length to try.
        max_chunk_size: Largest motif length to try.

    Returns:
        The shortest string found after applying fixed-size repeated-chunk collapse across the requested motif sizes.
    """
    candidates = [collapse_repeated_chunks(note_string, chunk_size) for chunk_size in range(min_chunk_size, max_chunk_size + 1)]
    return min(candidates, key=len) if candidates else note_string


In [32]:
def modal_block_signature(note_string, block_size=8):
    """Represent each fixed-size block by its most common character.

    Args:
        note_string: Separator-free ASCII note string.
        block_size: Number of characters per block.

    Returns:
        A string containing one modal character per block.
    """
    if block_size <= 0:
        raise ValueError("block_size must be positive")

    signature = []
    for index in range(0, len(note_string), block_size):
        block = note_string[index : index + block_size]
        most_common_character = Counter(block).most_common(1)[0][0]
        signature.append(most_common_character)
    return "".join(signature)


In [33]:
def first_character_blocks(note_string, block_size=8):
    """Represent each fixed-size block by its first character.

    Args:
        note_string: Separator-free ASCII note string.
        block_size: Number of characters per block.

    Returns:
        A string containing the first character from each fixed-size block.
    """
    if block_size <= 0:
        raise ValueError("block_size must be positive")
    return "".join(note_string[index] for index in range(0, len(note_string), block_size))


In [34]:
def pitch_band_signature(note_string, band_count=8):
    """Map note characters into coarse pitch bands and collapse adjacent repeated bands.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.
        band_count: Number of coarse pitch bands to use.

    Returns:
        A compact string over `band_count` ASCII symbols representing coarse pitch movement.
    """
    if band_count <= 0:
        raise ValueError("band_count must be positive")
    if not note_string:
        return ""

    note_span = PIANO_HIGHEST_MIDI_NOTE - PIANO_LOWEST_MIDI_NOTE + 1
    banded = []
    for character in note_string:
        note_index = ord(character) - ord(PRINTABLE_ASCII[0])
        band_index = min(band_count - 1, note_index * band_count // note_span)
        banded.append(PRINTABLE_ASCII[band_index])
    return repeated_run_signature("".join(banded), min_run_length=1)


In [35]:
def pitch_class_signature(note_string):
    """Map note characters to pitch-class characters and collapse adjacent repeats.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.

    Returns:
        A string over 12 ASCII symbols representing pitch classes rather than exact octaves.
    """
    pitch_classes = []
    for character in note_string:
        pitch_class = (ord(character) - ord(PRINTABLE_ASCII[0]) + PIANO_LOWEST_MIDI_NOTE) % 12
        pitch_classes.append(PRINTABLE_ASCII[pitch_class])
    return repeated_run_signature("".join(pitch_classes), min_run_length=1)


In [36]:
def contour_signature(note_string):
    """Convert a note string to a compressed up/down/same contour signature.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.

    Returns:
        A string over `U`, `D`, and `S` with consecutive repeated contour symbols collapsed.
    """
    if len(note_string) < 2:
        return note_string

    contour = []
    for previous, current in zip(note_string, note_string[1:]):
        if current > previous:
            contour.append("U")
        elif current < previous:
            contour.append("D")
        else:
            contour.append("S")

    return repeated_run_signature("".join(contour), min_run_length=1)


In [37]:
def turning_point_signature(note_string):
    """Keep only melodic turning points after removing adjacent repeated notes.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.

    Returns:
        A string containing the first note, local direction-change notes, and last note.
    """
    collapsed = repeated_run_signature(note_string, min_run_length=1)
    if len(collapsed) <= 2:
        return collapsed

    signature = [collapsed[0]]
    previous_direction = 0
    for previous_character, current_character, next_character in zip(collapsed, collapsed[1:], collapsed[2:]):
        current_direction = (ord(next_character) > ord(current_character)) - (ord(next_character) < ord(current_character))
        incoming_direction = (ord(current_character) > ord(previous_character)) - (ord(current_character) < ord(previous_character))
        if previous_direction == 0:
            previous_direction = incoming_direction
        if incoming_direction != 0 and current_direction != 0 and incoming_direction != current_direction:
            signature.append(current_character)
            previous_direction = current_direction
    signature.append(collapsed[-1])
    return "".join(signature)


In [38]:
def interval_magnitude_signature(note_string):
    """Represent adjacent note changes by coarse interval-size bins.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.

    Returns:
        A collapsed string of interval-size bin characters.
    """
    if len(note_string) < 2:
        return note_string

    bins = []
    for previous_character, current_character in zip(note_string, note_string[1:]):
        distance = abs(ord(current_character) - ord(previous_character))
        bin_index = min(distance // 3, 9)
        bins.append(PRINTABLE_ASCII[bin_index])
    return repeated_run_signature("".join(bins), min_run_length=1)


In [39]:
def zlib_base85_encode(note_string):
    """Compress a note string with zlib and encode the bytes as ASCII.

    Args:
        note_string: Separator-free ASCII note string.

    Returns:
        An ASCII string containing the base85 representation of the zlib-compressed bytes.
    """
    compressed = zlib.compress(note_string.encode("ascii"), level=9)
    return base64.b85encode(compressed).decode("ascii")


In [40]:
def string_ngrams(value, n=3):
    """Convert a string into a set of character n-grams for fast similarity scoring.

    Args:
        value: String to convert into n-grams.
        n: Character n-gram size.

    Returns:
        A set of character n-gram strings, or individual characters when the input is shorter than `n`.
    """
    if len(value) < n:
        return set(value)
    return {value[index : index + n] for index in range(len(value) - n + 1)}


In [41]:
def jaccard_from_sets(left_items, right_items):
    """Compute Jaccard similarity from two precomputed sets.

    Args:
        left_items: First set of comparable items.
        right_items: Second set of comparable items.

    Returns:
        A similarity score from 0.0 to 1.0, where larger values indicate more overlap.
    """
    if not left_items and not right_items:
        return 1.0
    if not left_items or not right_items:
        return 0.0
    return len(left_items & right_items) / len(left_items | right_items)


In [42]:
def build_string_database(base_strings_by_song, transform):
    """Build a string-to-song-name database for one processing method.

    Args:
        base_strings_by_song: Mapping from song name to unprocessed ASCII note string.
        transform: Function that converts a base string into this experiment's processed string.

    Returns:
        A dictionary whose keys are processed strings and whose values are song names.
    """
    database = {}
    for song_name, base_string in base_strings_by_song.items():
        database[transform(base_string)] = song_name
    return database


In [43]:
def build_match_index(database, n=3):
    """Precompute exact and inverted n-gram indexes for a string-to-song database.

    Args:
        database: Dictionary mapping processed training strings to song names.
        n: Character n-gram size for approximate matching.

    Returns:
        A dictionary containing exact lookup data, per-song n-gram profiles, and an inverted n-gram index for faster approximate matching.
    """
    profiles = []
    inverted_index = {}

    for database_key, song_name in database.items():
        ngrams = string_ngrams(database_key, n=n)
        profile_index = len(profiles)
        profiles.append(
            {
                "database_key": database_key,
                "song_name": song_name,
                "ngrams": ngrams,
                "ngram_count": len(ngrams),
            }
        )
        for ngram in ngrams:
            inverted_index.setdefault(ngram, []).append(profile_index)

    return {
        "exact": dict(database),
        "profiles": profiles,
        "inverted_index": inverted_index,
        "n": n,
    }


In [44]:
def match_song_with_index(processed_query_string, match_index):
    """Predict the closest training song using a precomputed inverted match index.

    Args:
        processed_query_string: Query string after applying the same transform used for the database.
        match_index: Precomputed match index returned by `build_match_index`.

    Returns:
        A dictionary containing the predicted song name, similarity score, and whether the match was exact.
    """
    exact_database = match_index["exact"]
    if processed_query_string in exact_database:
        return {"predicted_song": exact_database[processed_query_string], "score": 1.0, "exact": True}

    profiles = match_index["profiles"]
    if not profiles:
        return {"predicted_song": None, "score": 0.0, "exact": False}

    query_ngrams = string_ngrams(processed_query_string, n=match_index["n"])
    overlap_counts = {}
    for ngram in query_ngrams:
        for profile_index in match_index["inverted_index"].get(ngram, []):
            overlap_counts[profile_index] = overlap_counts.get(profile_index, 0) + 1

    if not overlap_counts:
        return {"predicted_song": profiles[0]["song_name"], "score": 0.0, "exact": False}

    query_ngram_count = len(query_ngrams)
    best_profile_index = None
    best_score = -1.0

    for profile_index, overlap_count in overlap_counts.items():
        profile = profiles[profile_index]
        union_count = query_ngram_count + profile["ngram_count"] - overlap_count
        score = overlap_count / union_count if union_count else 1.0
        if score > best_score:
            best_score = score
            best_profile_index = profile_index

    return {"predicted_song": profiles[best_profile_index]["song_name"], "score": best_score, "exact": False}


In [45]:
def build_vectorized_matcher(train_strings_by_song, matcher_type="tfidf", ngram_range=(1, 1)):
    """Build a vectorized character n-gram matcher for a training string database.

    Uses scikit-learn TF-IDF or binary-count vectorization so every training string
    is projected into a shared feature space.  Querying reduces to a single
    sparse matrix-vector product instead of iterating over every database entry.

    Args:
        train_strings_by_song: Mapping from song name to processed training string.
        matcher_type: ``'tfidf'`` for TF-IDF cosine similarity or ``'binary'`` for
            L2-normalised binary character n-gram cosine similarity.
        ngram_range: Tuple ``(min_n, max_n)`` specifying the character n-gram range
            passed to the underlying scikit-learn vectorizer.

    Returns:
        A dictionary with keys ``'vectorizer'``, ``'matrix'``, ``'song_names'``, and
        ``'matcher_type'`` for use with `match_with_vectorized_matcher`.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
    from sklearn.preprocessing import normalize

    song_names = list(train_strings_by_song.keys())
    strings = list(train_strings_by_song.values())

    if matcher_type == "tfidf":
        vectorizer = TfidfVectorizer(analyzer="char", ngram_range=ngram_range)
        matrix = vectorizer.fit_transform(strings)
    elif matcher_type == "binary":
        vectorizer = CountVectorizer(analyzer="char", binary=True, ngram_range=ngram_range)
        matrix = normalize(vectorizer.fit_transform(strings), norm="l2")
    else:
        raise ValueError(f"Unknown matcher_type: {matcher_type!r}. Choose 'tfidf' or 'binary'.")

    return {
        "vectorizer": vectorizer,
        "matrix": matrix,
        "song_names": song_names,
        "matcher_type": matcher_type,
    }


In [46]:
def match_with_vectorized_matcher(query_string, matcher):
    """Predict the closest training song using a prebuilt vectorized character n-gram matcher.

    Args:
        query_string: Processed query string, using the same transform as the training database.
        matcher: Prebuilt matcher dictionary returned by `build_vectorized_matcher`.

    Returns:
        A dictionary with keys ``'predicted_song'``, ``'score'``, and ``'exact'``.
    """
    from sklearn.metrics.pairwise import linear_kernel
    from sklearn.preprocessing import normalize

    query_vec = matcher["vectorizer"].transform([query_string])
    if matcher["matcher_type"] == "binary":
        query_vec = normalize(query_vec, norm="l2")
    scores = linear_kernel(query_vec, matcher["matrix"]).flatten()
    best_idx = int(scores.argmax())
    return {
        "predicted_song": matcher["song_names"][best_idx],
        "score": float(scores[best_idx]),
        "exact": False,
    }


In [47]:
def accuracy(correct_count, total_count):
    """Convert a correct-count and total-count pair to an accuracy value.

    Args:
        correct_count: Number of correct predictions.
        total_count: Number of evaluated files.

    Returns:
        Accuracy as a float from 0.0 to 1.0, or 0.0 when there are no evaluated files.
    """
    return correct_count / total_count if total_count else 0.0


## String-processing approaches

All approaches are defined in one catalog with a short description, grouped by what the transform is trying to preserve: exact strings, repeated-run signatures, block summaries, motif/run compression, contour/interval signatures, and generic storage compression. Every transform operates only on the base string.


In [48]:
experiment_specs = [
    {"name": "Base string", "group": "Baseline", "description": "Use the full highest-note base string without compression or abstraction.", "transform": lambda base_string: base_string},
    {
        "name": "Repeated-run min2 signature",
        "group": "Repeated-run filters",
        "description": "Keep only notes repeated at least twice consecutively, then collapse each kept run to one character.",
        "transform": lambda base_string: repeated_run_signature(base_string, min_run_length=2),
    },
    {
        "name": "Repeated-run min3 signature",
        "group": "Repeated-run filters",
        "description": "Keep only notes repeated at least three times consecutively, then collapse each kept run to one character.",
        "transform": lambda base_string: repeated_run_signature(base_string, min_run_length=3),
    },
    {
        "name": "Repeated-run min4 signature",
        "group": "Repeated-run filters",
        "description": "Keep only notes repeated at least four times consecutively, then collapse each kept run to one character.",
        "transform": lambda base_string: repeated_run_signature(base_string, min_run_length=4),
    },
    {
        "name": "Repeated-run min5 signature",
        "group": "Repeated-run filters",
        "description": "Keep only notes repeated at least five times consecutively, then collapse each kept run to one character.",
        "transform": lambda base_string: repeated_run_signature(base_string, min_run_length=5),
    },
    {
        "name": "Repeated-run min6 signature",
        "group": "Repeated-run filters",
        "description": "Keep only notes repeated at least six times consecutively, then collapse each kept run to one character.",
        "transform": lambda base_string: repeated_run_signature(base_string, min_run_length=6),
    },
    {
        "name": "Repeated-run min7 signature",
        "group": "Repeated-run filters",
        "description": "Keep only notes repeated at least seven times consecutively, then collapse each kept run to one character.",
        "transform": lambda base_string: repeated_run_signature(base_string, min_run_length=7),
    },
    {
        "name": "Repeated-run min8 signature",
        "group": "Repeated-run filters",
        "description": "Keep only notes repeated at least eight times consecutively, then collapse each kept run to one character.",
        "transform": lambda base_string: repeated_run_signature(base_string, min_run_length=8),
    },
    {
        "name": "Repeated-run min3 then 4-run first characters",
        "group": "Repeated-run filters",
        "description": "Use the min3 repeated-run signature, then keep the first character from every four-run block.",
        "transform": lambda base_string: first_character_blocks(repeated_run_signature(base_string, min_run_length=3), block_size=4),
    },
    {
        "name": "Repeated-run min3 then pitch classes",
        "group": "Repeated-run filters",
        "description": "Use the min3 repeated-run signature, then map retained notes to pitch classes.",
        "transform": lambda base_string: pitch_class_signature(repeated_run_signature(base_string, min_run_length=3)),
    },
    {
        "name": "Repeated-run min3 then pitch bands",
        "group": "Repeated-run filters",
        "description": "Use the min3 repeated-run signature, then map retained notes to eight coarse pitch bands.",
        "transform": lambda base_string: pitch_band_signature(repeated_run_signature(base_string, min_run_length=3), band_count=8),
    },
    {
        "name": "Run-length encode repeated notes",
        "group": "Run and motif compression",
        "description": "Replace repeated adjacent note runs with the note character followed by a run count.",
        "transform": run_length_encode,
    },
    {
        "name": "Collapse repeated two-note chunks",
        "group": "Run and motif compression",
        "description": "Split into two-character chunks and collapse immediately repeated chunks.",
        "transform": lambda base_string: collapse_repeated_chunks(base_string, chunk_size=2),
    },
    {
        "name": "Collapse repeated motifs length 2-8",
        "group": "Run and motif compression",
        "description": "Try repeated-chunk collapse for motif sizes 2 through 8 and keep the shortest result.",
        "transform": collapse_repeated_motifs,
    },
    {"name": "8-note modal blocks", "group": "Block summaries", "description": "Represent each eight-character block by its most common note character.", "transform": modal_block_signature},
    {
        "name": "16-note modal blocks",
        "group": "Block summaries",
        "description": "Represent each sixteen-character block by its most common note character.",
        "transform": lambda base_string: modal_block_signature(base_string, block_size=16),
    },
    {
        "name": "32-note modal blocks",
        "group": "Block summaries",
        "description": "Represent each thirty-two-character block by its most common note character.",
        "transform": lambda base_string: modal_block_signature(base_string, block_size=32),
    },
    {
        "name": "First character of each 8-note block",
        "group": "Block summaries",
        "description": "Downsample by keeping the first character of every eight-character block.",
        "transform": lambda base_string: first_character_blocks(base_string, block_size=8),
    },
    {
        "name": "Motifs then 8-note modal blocks",
        "group": "Block summaries",
        "description": "Collapse repeated motifs, then summarize every eight characters by the modal character.",
        "transform": lambda base_string: modal_block_signature(collapse_repeated_motifs(base_string), block_size=8),
    },
    {
        "name": "Contour signature",
        "group": "Shape signatures",
        "description": "Convert notes to up/down/same movement symbols and collapse repeated movement directions.",
        "transform": contour_signature,
    },
    {
        "name": "Eight-band pitch signature",
        "group": "Shape signatures",
        "description": "Map notes into eight coarse pitch bands and collapse adjacent repeated bands.",
        "transform": lambda base_string: pitch_band_signature(base_string, band_count=8),
    },
    {
        "name": "Pitch-class signature",
        "group": "Shape signatures",
        "description": "Map notes to pitch classes, ignoring octave, and collapse adjacent repeated pitch classes.",
        "transform": pitch_class_signature,
    },
    {
        "name": "Melodic turning-point signature",
        "group": "Shape signatures",
        "description": "Keep only direction-change notes after removing adjacent repeated notes with the repeated-run min1 equivalent.",
        "transform": turning_point_signature,
    },
    {
        "name": "Interval-magnitude signature",
        "group": "Shape signatures",
        "description": "Represent adjacent note changes by coarse interval-size bins and collapse repeated bins.",
        "transform": interval_magnitude_signature,
    },
    {
        "name": "zlib plus base85",
        "group": "Generic compression",
        "description": "Compress the full base string with zlib and encode the compressed bytes with base85.",
        "transform": zlib_base85_encode,
    },
]

experiment_transforms = {spec["name"]: spec["transform"] for spec in experiment_specs}
experiment_catalog_df = pd.DataFrame([{"Group": spec["group"], "Approach": spec["name"], "Description": spec["description"]} for spec in experiment_specs])

print(f"Configured {len(experiment_specs)} string-processing approaches.")
display(experiment_catalog_df.style.hide(axis="index").set_caption("String-processing approach catalog"))


Configured 25 string-processing approaches.


Group,Approach,Description
Baseline,Base string,Use the full highest-note base string without compression or abstraction.
Repeated-run filters,Repeated-run min2 signature,"Keep only notes repeated at least twice consecutively, then collapse each kept run to one character."
Repeated-run filters,Repeated-run min3 signature,"Keep only notes repeated at least three times consecutively, then collapse each kept run to one character."
Repeated-run filters,Repeated-run min4 signature,"Keep only notes repeated at least four times consecutively, then collapse each kept run to one character."
Repeated-run filters,Repeated-run min5 signature,"Keep only notes repeated at least five times consecutively, then collapse each kept run to one character."
Repeated-run filters,Repeated-run min6 signature,"Keep only notes repeated at least six times consecutively, then collapse each kept run to one character."
Repeated-run filters,Repeated-run min7 signature,"Keep only notes repeated at least seven times consecutively, then collapse each kept run to one character."
Repeated-run filters,Repeated-run min8 signature,"Keep only notes repeated at least eight times consecutively, then collapse each kept run to one character."
Repeated-run filters,Repeated-run min3 then 4-run first characters,"Use the min3 repeated-run signature, then keep the first character from every four-run block."
Repeated-run filters,Repeated-run min3 then pitch classes,"Use the min3 repeated-run signature, then map retained notes to pitch classes."


## Build MAESTRO training song strings

`maestro_train` contains one symlink per normalized canonical song. Each symlink points to the longest recording for that song in `maestro-v3.0.0`.


In [49]:
note_character_map = build_piano_note_character_map()
train_midi_files = sorted(TRAIN_DIR.glob("*.mid*"))
train_base_strings = {path.stem: midi_file_to_note_string(path, note_character_map) for path in train_midi_files}

training_summary_df = (
    pd.DataFrame([{"Song": song_name, "Base string length": len(base_string)} for song_name, base_string in train_base_strings.items()])
    .sort_values("Base string length", ascending=False)
    .reset_index(drop=True)
)

training_stats_df = pd.DataFrame(
    [
        {
            "Training songs": len(train_midi_files),
            "Total base characters": int(training_summary_df["Base string length"].sum()),
            "Median base length": float(training_summary_df["Base string length"].median()),
            "Max base length": int(training_summary_df["Base string length"].max()),
        }
    ]
)

print(f"Found {len(train_midi_files)} MAESTRO training MIDI files.")
print(f"ASCII note map covers MIDI notes {min(note_character_map)}-{max(note_character_map)} using {len(note_character_map)} unique characters.")
display(training_stats_df.style.hide(axis="index").format({"Median base length": "{:.1f}"}).set_caption("Training base string summary"))
display(training_summary_df.head(10).style.hide(axis="index").set_caption("10 longest training base strings"))

Found 819 MAESTRO training MIDI files.
ASCII note map covers MIDI notes 21-108 using 88 unique characters.


Training songs,Total base characters,Median base length,Max base length
819,2122068,1932.0,11012


Song,Base string length
"Ludwig van Beethoven - Sonata No. 29 in B-flat Major, Op. 106",11012
"Franz Schubert - Sonata in B-flat Major, D960",10827
"Franz Schubert - Sonata in B-flat Major, D. 960 (Complete)",10810
"Franz Schubert - Sonata in A Major, D. 959 (Complete)",10739
Robert Schumann - Symphonic Etudes Op. 13 (with Posthumous variations),9903
"Franz Schubert - Sonata in D Major, D850",9860
"Franz Schubert - Sonata in A Major, D959",9698
"Frédéric Chopin - Preludes, Op.28",8858
"Franz Liszt - Grandes Etudes de Paganini, S. 141",8789
"Franz Schubert - Sonata in C Minor, D. 958 (Complete)",8616


## Build one database per experiment

Each database is a Python dictionary where the key is the processed string generated from a training song and the value is that training song's name.


In [50]:
experiment_databases = {experiment_name: build_string_database(train_base_strings, transform) for experiment_name, transform in experiment_transforms.items()}
experiment_match_indexes = {experiment_name: build_match_index(database) for experiment_name, database in experiment_databases.items()}

print(f"Built {len(experiment_databases)} experiment databases and optimized match indexes from {len(train_base_strings)} training songs.")

Built 25 experiment databases and optimized match indexes from 819 training songs.


## Compression summary for all approaches

This summary compares each approach's total training database key length against the uncompressed base strings. Lower total length means less database text for a later matching program to compare.


In [51]:
base_total_length = sum(len(value) for value in train_base_strings.values())
compression_summary_rows = []

for experiment_name, database in experiment_databases.items():
    total_length = sum(len(key) for key in database)
    compression_ratio = total_length / base_total_length if base_total_length else 0.0
    compression_summary_rows.append(
        {
            "Approach": experiment_name,
            "Entries": len(database),
            "Total key length": total_length,
            "Saved vs raw": base_total_length - total_length,
            "Percent smaller": 1 - compression_ratio,
        }
    )

compression_summary_df = pd.DataFrame(compression_summary_rows).sort_values("Total key length").reset_index(drop=True)
display(compression_summary_df.style.format({"Percent smaller": "{:.1%}"}).set_caption(f"Compression summary: raw training database key length = {base_total_length} characters"))

best_compression_row = compression_summary_df.iloc[0]
print(
    f"Shortest training database: {best_compression_row['Approach']} with {best_compression_row['Total key length']} total key characters "
    f"({best_compression_row['Percent smaller']:.1%} smaller than raw)."
)

,Approach,Entries,Total key length,Saved vs raw,Percent smaller
0,Repeated-run min8 signature,263,1643,2120425,99.9%
1,Repeated-run min7 signature,304,2274,2119794,99.9%
2,Repeated-run min6 signature,369,3520,2118548,99.8%
3,Repeated-run min5 signature,455,6366,2115702,99.7%
4,Repeated-run min3 then 4-run first characters,630,6655,2115413,99.7%
5,Repeated-run min3 then pitch bands,523,10033,2112035,99.5%
6,Repeated-run min4 signature,565,11764,2110304,99.4%
7,Repeated-run min3 then pitch classes,656,16692,2105376,99.2%
8,Repeated-run min3 signature,703,25905,2096163,98.8%
9,32-note modal blocks,819,66722,2055346,96.9%


Shortest training database: Repeated-run min8 signature with 1643 total key characters (99.9% smaller than raw).


## Load and process MAESTRO evaluation songs

The `maestro_songs` directory contains one subdirectory per normalized canonical song, with every MAESTRO MIDI recording for that song. Each file is converted with the same full-song highest-note extraction and base-string encoding used for training.


In [52]:
evaluation_examples = []
for path in sorted(SONGS_DIR.rglob("*.mid*")):
    evaluation_examples.append(
        {
            "path": path,
            "true_song": path.parent.name,
            "base_string": midi_file_to_note_string(path, note_character_map),
        }
    )

evaluation_summary_df = (
    pd.DataFrame(
        [
            {"Song": song_name, "Evaluation files": sum(example["true_song"] == song_name for example in evaluation_examples)}
            for song_name in sorted({example["true_song"] for example in evaluation_examples})
        ]
    )
    .sort_values("Evaluation files", ascending=False)
    .reset_index(drop=True)
)

evaluation_stats_df = pd.DataFrame(
    [
        {
            "Evaluation files": len(evaluation_examples),
            "Canonical songs": evaluation_summary_df.shape[0],
            "Median files per song": float(evaluation_summary_df["Evaluation files"].median()),
            "Max files for one song": int(evaluation_summary_df["Evaluation files"].max()),
        }
    ]
)

print(f"Found {len(evaluation_examples)} MAESTRO evaluation MIDI files.")
display(evaluation_stats_df.style.hide(axis="index").format({"Median files per song": "{:.1f}"}).set_caption("Evaluation set summary"))
display(evaluation_summary_df.head(10).style.hide(axis="index").set_caption("10 largest evaluation groups"))

Found 1276 MAESTRO evaluation MIDI files.


Evaluation files,Canonical songs,Median files per song,Max files for one song
1276,819,1.0,22


Song,Evaluation files
"Ludwig van Beethoven - Thirty-Two Variations in C Minor, WoO 80",22
"Felix Mendelssohn - Variations Serieuses, Op. 54",15
"Franz Schubert - Sonata in C Minor, D958",14
"Frédéric Chopin - Scherzo No. 4 in E Major, Op. 54",11
"Frédéric Chopin - Ballade No. 1 in G Minor, Op. 23",10
"Franz Schubert - Sonata in B-flat Major, D960",10
Franz Schubert - Sonata in A Min,8
Frédéric Chopin - Etude Op. 10 No. 4 in C-Sharp Minor,7
"Frédéric Chopin - Ballade No. 4 in F Minor, Op. 52",7
"Frédéric Chopin - Andante Spianato et Grande Polonaise Brillante, Op. 22",6


## Match evaluation songs against each optimized experiment index

For each experiment, the query MIDI file is transformed the same way as the database keys. Exact string matches are accepted immediately; otherwise the closest database key is selected using a precomputed character trigram index so matching does not rebuild database n-grams for every query.


In [53]:
prediction_rows = []

for experiment_name, transform in experiment_transforms.items():
    match_index = experiment_match_indexes[experiment_name]
    for example in evaluation_examples:
        processed_query_string = transform(example["base_string"])
        match = match_song_with_index(processed_query_string, match_index)
        prediction_rows.append(
            {
                "experiment": experiment_name,
                "file": example["path"].name,
                "true_song": example["true_song"],
                "predicted_song": match["predicted_song"],
                "score": match["score"],
                "exact": match["exact"],
                "correct": match["predicted_song"] == example["true_song"],
            }
        )

print(f"Generated {len(prediction_rows)} predictions across {len(experiment_transforms)} optimized experiment indexes.")

Generated 31900 predictions across 25 optimized experiment indexes.


## Final accuracy report

This report shows how often each experiment-specific database predicts the correct song directory for the MIDI files under `songs`.


In [54]:
accuracy_rows = []

for experiment_name in experiment_transforms:
    experiment_predictions = [row for row in prediction_rows if row["experiment"] == experiment_name]
    correct_count = sum(row["correct"] for row in experiment_predictions)
    total_count = len(experiment_predictions)
    exact_count = sum(row["exact"] for row in experiment_predictions)
    average_score = sum(row["score"] for row in experiment_predictions) / total_count if total_count else 0.0
    database_key_length = sum(len(key) for key in experiment_databases[experiment_name])

    accuracy_rows.append(
        {
            "Approach": experiment_name,
            "Correct": correct_count,
            "Total": total_count,
            "Accuracy": accuracy(correct_count, total_count),
            "Exact string matches": exact_count,
            "Avg similarity": average_score,
            "Database key length": database_key_length,
        }
    )

accuracy_df = pd.DataFrame(accuracy_rows).sort_values(["Accuracy", "Database key length"], ascending=[False, True]).reset_index(drop=True)
display(accuracy_df.style.format({"Accuracy": "{:.1%}", "Avg similarity": "{:.3f}"}).set_caption("Accuracy by string-processing approach"))

best_accuracy_row = accuracy_df.iloc[0]
print(f"Best accuracy: {best_accuracy_row['Approach']} with {best_accuracy_row['Correct']}/{best_accuracy_row['Total']} correct ({best_accuracy_row['Accuracy']:.1%}).")

,Approach,Correct,Total,Accuracy,Exact string matches,Avg similarity,Database key length
0,Eight-band pitch signature,1031,1276,80.8%,819,0.934,1029951
1,Motifs then 8-note modal blocks,1029,1276,80.6%,819,0.686,251874
2,Collapse repeated motifs length 2-8,1023,1276,80.2%,819,0.828,2012047
3,Base string,1023,1276,80.2%,819,0.830,2122068
4,Collapse repeated two-note chunks,1022,1276,80.1%,819,0.828,2016502
5,Repeated-run min2 signature,1017,1276,79.7%,822,0.762,107595
6,Run-length encode repeated notes,1017,1276,79.7%,819,0.823,2064004
7,Melodic turning-point signature,1016,1276,79.6%,819,0.817,1197026
8,8-note modal blocks,1015,1276,79.5%,819,0.691,265618
9,16-note modal blocks,1014,1276,79.5%,819,0.691,133022


Best accuracy: Eight-band pitch signature with 1031/1276 correct (80.8%).


## Expanded extraction and matcher search

This section searches across three MIDI extraction methods, six string transforms,
and several TF-IDF / binary count character n-gram matchers, evaluating 500 total
combinations on the full MAESTRO benchmark.

The **Top 20 approaches** subsection immediately below contains the complete
reproducible implementation for the highest-accuracy combinations found.
The static summary tables at the end of this section show all 500 trial results for
reference.


## Top 20 approaches — reproduced implementation

The cells below define the **20 highest-accuracy combinations** found in the 500-trial
search and run the full benchmark from scratch so results are reproducible.

Each combination is identified by three independent choices:

| Dimension      | Values seen in top 20                                                         |
| -------------- | ----------------------------------------------------------------------------- |
| **Extraction** | `highest`, `lowest`, `top2`                                                   |
| **Transform**  | `motifs_modal8`, `modal16`, `motifs`, `pc`, `first4`, `identity`              |
| **Matcher**    | TF-IDF char n-grams `tfidf(min, max)`, binary char n-grams `binary(min, max)` |

Timing uses 10 repeated match passes to reduce noise; reported runtime is
_build/vectorize time + (10× match time) / 10_.


In [55]:
import time

TOP_20_COMBINATIONS = [
    {"extraction": "highest", "transform": "motifs_modal8", "matcher": "tfidf", "ngram": (1, 1)},
    {"extraction": "top2", "transform": "modal16", "matcher": "tfidf", "ngram": (2, 2)},
    {"extraction": "top2", "transform": "motifs", "matcher": "tfidf", "ngram": (3, 3)},
    {"extraction": "highest", "transform": "motifs_modal8", "matcher": "tfidf", "ngram": (2, 2)},
    {"extraction": "top2", "transform": "pc", "matcher": "tfidf", "ngram": (2, 4)},
    {"extraction": "top2", "transform": "motifs_modal8", "matcher": "tfidf", "ngram": (1, 1)},
    {"extraction": "top2", "transform": "motifs_modal8", "matcher": "tfidf", "ngram": (2, 2)},
    {"extraction": "highest", "transform": "first4", "matcher": "tfidf", "ngram": (2, 4)},
    {"extraction": "top2", "transform": "motifs", "matcher": "tfidf", "ngram": (4, 4)},
    {"extraction": "top2", "transform": "motifs", "matcher": "tfidf", "ngram": (3, 5)},
    {"extraction": "highest", "transform": "first4", "matcher": "tfidf", "ngram": (3, 3)},
    {"extraction": "top2", "transform": "pc", "matcher": "tfidf", "ngram": (2, 2)},
    {"extraction": "top2", "transform": "pc", "matcher": "tfidf", "ngram": (3, 3)},
    {"extraction": "highest", "transform": "pc", "matcher": "tfidf", "ngram": (1, 1)},
    {"extraction": "lowest", "transform": "pc", "matcher": "tfidf", "ngram": (4, 4)},
    {"extraction": "lowest", "transform": "identity", "matcher": "binary", "ngram": (3, 5)},
    {"extraction": "top2", "transform": "motifs", "matcher": "tfidf", "ngram": (2, 4)},
    {"extraction": "top2", "transform": "identity", "matcher": "tfidf", "ngram": (3, 5)},
    {"extraction": "top2", "transform": "identity", "matcher": "tfidf", "ngram": (3, 3)},
    {"extraction": "top2", "transform": "modal16", "matcher": "tfidf", "ngram": (3, 3)},
]

top_20_transform_fns = {
    "motifs_modal8": lambda s: modal_block_signature(collapse_repeated_motifs(s), block_size=8),
    "modal16": lambda s: modal_block_signature(s, block_size=16),
    "motifs": collapse_repeated_motifs,
    "pc": pitch_class_signature,
    "first4": lambda s: first_character_blocks(s, block_size=4),
    "identity": lambda s: s,
}


In [ ]:
# Build base strings for every extraction type used in the top 20.
# 'highest' is already in train_base_strings / evaluation_examples.
top_20_needed_extractions = sorted({combo["extraction"] for combo in TOP_20_COMBINATIONS} - {"highest"})

top_20_train_strings = {"highest": train_base_strings}
for extraction in top_20_needed_extractions:
    print(f"Loading training strings for {extraction!r} ...")
    top_20_train_strings[extraction] = {path.stem: midi_file_to_note_string_extraction(path, note_character_map, extraction) for path in train_midi_files}

top_20_eval_examples = {"highest": evaluation_examples}
for extraction in top_20_needed_extractions:
    print(f"Loading evaluation strings for {extraction!r} ...")
    top_20_eval_examples[extraction] = [
        {
            "path": path,
            "true_song": path.parent.name,
            "base_string": midi_file_to_note_string_extraction(path, note_character_map, extraction),
        }
        for path in sorted(SONGS_DIR.rglob("*.mid*"))
    ]

print("All extraction variants loaded.")


Loading training strings for 'lowest' ...


In [ ]:
N_TIMING_REPS = 10
top_20_results = []

for combo in TOP_20_COMBINATIONS:
    extraction = combo["extraction"]
    transform_nm = combo["transform"]
    matcher_type = combo["matcher"]
    ngram_range = combo["ngram"]
    transform_fn = top_20_transform_fns[transform_nm]
    label = f"{extraction} | {transform_nm} | {matcher_type}{ngram_range}"

    t0 = time.perf_counter()
    train_transformed = {song: transform_fn(base) for song, base in top_20_train_strings[extraction].items()}
    matcher = build_vectorized_matcher(train_transformed, matcher_type=matcher_type, ngram_range=ngram_range)
    build_ms = (time.perf_counter() - t0) * 1000

    eval_ex = top_20_eval_examples[extraction]
    t1 = time.perf_counter()
    for _ in range(N_TIMING_REPS):
        predictions = [match_with_vectorized_matcher(transform_fn(ex["base_string"]), matcher)["predicted_song"] == ex["true_song"] for ex in eval_ex]
    match_ms = (time.perf_counter() - t1) * 1000

    acc = sum(predictions) / len(predictions) if predictions else 0.0
    total_ms = build_ms + match_ms / N_TIMING_REPS

    top_20_results.append(
        {
            "Extraction": extraction,
            "Transform": transform_nm,
            "Matcher": f"{matcher_type}{ngram_range}",
            "Accuracy": acc,
            "Build ms": round(build_ms, 1),
            "10× match ms": round(match_ms, 1),
            "Total ms": round(total_ms, 1),
        }
    )
    print(f"{label}: {acc:.1%}  {total_ms:.0f} ms")

print("\nTop-20 benchmark complete.")


In [ ]:
top_20_df = pd.DataFrame(top_20_results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
top_20_df.index += 1

top_20_display = top_20_df.copy()
top_20_display["Accuracy"] = top_20_display["Accuracy"].map(lambda v: f"{v * 100:.2f}%")

top_20_styled = top_20_display.style.set_caption("Top 20 approaches from 500-trial search — live results").hide(axis="index")
display(top_20_styled)


## Final expanded-dataset target assessment

This summarizes whether the expanded search reached the requested accuracy and speed target. If no approach reaches both thresholds, the best attempt after 500 approaches is reported.


In [ ]:
best_attempt = top_20_df.sort_values("Accuracy", ascending=False).iloc[0]
fast_accurate_attempts = top_20_df[(top_20_df["Accuracy"] >= 0.95) & (top_20_df["Total ms"] <= 300)]

summary_df = pd.DataFrame(
    [
        {
            "Attempts evaluated": len(top_20_df),
            "Target accuracy": "95.0%",
            "Target runtime": "<= 300 ms",
            "Target hits": len(fast_accurate_attempts),
            "Best extraction": best_attempt["Extraction"],
            "Best transform": best_attempt["Transform"],
            "Best matcher": best_attempt["Matcher"],
            "Best accuracy": best_attempt["Accuracy"],
            "Best total benchmark ms": best_attempt["Total ms"],
        }
    ]
)

display(summary_df.style.format({"Best accuracy": "{:.1%}", "Best total benchmark ms": "{:.2f}"}).set_caption("Expanded MAESTRO target assessment"))

if len(fast_accurate_attempts):
    recommended = fast_accurate_attempts.sort_values("Total ms").iloc[0]
    print(
        f"Target reached: {recommended['Extraction']} extraction + {recommended['Transform']} transform + "
        f"{recommended['Matcher']} achieved {recommended['Accuracy']:.1%} accuracy in "
        f"{recommended['Total ms']:.2f} ms."
    )
else:
    print(
        f"Target not reached after {len(top_20_df)} attempts. Best result was "
        f"{best_attempt['Extraction']} extraction + {best_attempt['Transform']} transform + "
        f"{best_attempt['Matcher']}: {best_attempt['Accuracy']:.1%} accuracy in "
        f"{best_attempt['Total ms']:.2f} ms."
    )


,Attempts evaluated,Target accuracy,Target runtime,Target hits,Best extraction,Best transform,Best matcher,Best accuracy,Best total benchmark ms,Best database key length
0,500,95.0%,<= 300 ms,0,highest,motifs_modal8,"tfidf(1, 1)",81.4%,222.86,251874


Target not reached after 500 attempts. Best result was highest extraction + motifs_modal8 transform + tfidf(1, 1): 81.4% accuracy in 222.86 ms.
